In [1]:
# correlation analysis is your next critical fairness checkpoint, because even if group representation 
# looks balanced and statistical tests look fine, bias can still sneak in through correlated features — 
# what’s called proxy bias.

# To detect hidden relationships (direct or indirect) between sensitive attributes
# (Gender, Race, Age) and other features
# (Education, Experience_Years, Skills, Screening_Score, etc.)
# that could cause unintended discrimination during selection or model training.

# In short: correlation analysis finds where bias hides behind other variables.

# Even if your model doesn’t use “Gender” explicitly, a correlated variable like Experience_Years, Education, or Certifications could still encode it.
# That’s how proxy bias forms — the model “learns” demographic patterns indirectly.

# Example: If Gender correlates strongly with Screening_Score, the selection model could favor one gender even if the “gender” column isn’t used.

# If correlation > 0.3 (or < -0.3), that’s a moderate link. Then there is potential for proxy bias.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, pointbiserialr, pearsonr
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import MultiLabelBinarizer
import warnings

df = pd.read_csv("../../datasets/recruitment.csv")
df.drop(columns=["Candidate_ID"], inplace=True)
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 65], labels=['18-25', '26-35', '36-45', '46-55', '56-65'])
df['Selected'] = df['Selected'].map({'Yes': 1, 'No': 0})
df.drop(columns=["Age"], inplace=True)

# Multi-label encoding for Skills
mlb = MultiLabelBinarizer()
skills_df = df.copy()

skills_df['Skills'] = skills_df['Skills'].fillna('').str.lower().str.split(',')

skills_encoded = pd.DataFrame(mlb.fit_transform(skills_df['Skills']),
                              columns=[f"skill_{s.replace(' ', '_')}" for s in mlb.classes_])

skills_df = pd.concat([df.drop(columns=['Skills']), skills_encoded], axis=1)

# Encode categorical variables
df_encoded = skills_df.copy()

categorical_cols = ["Gender", "Race", "Age_Group", "Education", "Job_Role_Applied", "Certifications"]
for col in categorical_cols:
    df_encoded[col] = pd.factorize(df_encoded[col])[0]

skills_cols = [col for col in df_encoded.columns if col.startswith('skill_')]
numerical_cols = ["Experience_Years", "Screening_Score", "Selected"]

df_encoded = df_encoded[categorical_cols + numerical_cols + skills_cols]

df0 = df_encoded.copy()  # from your code above

# Identify columns
sensitive = ["Gender", "Race", "Age_Group"]
target = "Selected"

# Separate types
skill_cols   = [c for c in df0.columns if c.startswith("skill_")]
num_cols     = ["Experience_Years", "Screening_Score"]  # numeric features
cat_cols     = ["Gender", "Race", "Age_Group", "Education", "Job_Role_Applied", "Certifications"] + skill_cols

# Drop target from proxy search
proxy_cols = [c for c in df0.columns if c != target]

# ---- 1) Helper functions ----
def cramers_v(x, y):
    # x, y are categorical/integers
    tbl = pd.crosstab(x, y)
    if tbl.size == 0 or (tbl.values.sum() == 0):
        return np.nan, np.nan
    chi2, p, dof, _ = chi2_contingency(tbl, correction=False)
    n = tbl.values.sum()
    if n == 0:
        return np.nan, p
    phi2 = chi2 / n
    r, k = tbl.shape
    denom = min(k-1, r-1)
    if denom <= 0:
        return np.nan, p
    v = np.sqrt(phi2 / denom)
    return v, p

def point_biserial_from_binary_series(bin_series, numeric_series):
    # bin_series must be 0/1
    try:
        r, p = pointbiserialr(bin_series, numeric_series)
    except Exception:
        r, p = np.nan, np.nan
    return r, p

def ensure_binary(series):
    # Map any two-category categorical/integer to 0/1
    vals = pd.Series(series).dropna().unique()
    if len(vals) != 2:
        return None
    mapping = {vals[0]: 0, vals[1]: 1}
    return pd.Series(series).map(mapping)

# ---- 2) Build association results between sensitive and all other features ----
rows = []
for s in sensitive:
    s_series = df0[s]
    # For each candidate proxy
    for f in proxy_cols:
        if f == s: 
            continue

        f_series = df0[f]
        metr = np.nan; pval = np.nan; mtype = None

        # Case A: categorical vs categorical (incl. skills)
        if f in cat_cols:
            v, p = cramers_v(s_series, f_series)
            metr, pval, mtype = v, p, "cramers_v"

        # Case B: numeric vs categorical-binary (point-biserial)
        elif f in num_cols:
            # only meaningful if sensitive is binary
            s_bin = ensure_binary(s_series)
            if s_bin is not None:
                r, p = point_biserial_from_binary_series(s_bin, f_series)
                metr, pval, mtype = r, p, "point_biserial"
            else:
                # If sensitive has >2 categories, ANOVA/KW belongs in the statistical test section
                # For proxy screening, compute max pairwise point-biserial across one-vs-rest
                cats = s_series.dropna().unique()
                vals = []
                for c in cats:
                    one_vs_rest = (s_series == c).astype(int)
                    r, p = point_biserial_from_binary_series(one_vs_rest, f_series)
                    vals.append((abs(r), p))
                if vals:
                    best = max(vals, key=lambda t: t[0])
                    metr, pval, mtype = np.sign(best[0]) * best[0], best[1], "point_biserial_o-v-r"

        rows.append({
            "sensitive": s,
            "feature": f,
            "metric": mtype,
            "association": metr,
            "p_value": pval
        })

assoc_df = pd.DataFrame(rows)

# ---- 3) Multiple testing correction (FDR) ----
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    mask = assoc_df["p_value"].notna()
    rej, p_adj, _, _ = multipletests(assoc_df.loc[mask, "p_value"], alpha=0.05, method="fdr_bh")
    assoc_df.loc[mask, "p_adj"] = p_adj
    assoc_df.loc[mask, "significant_fdr_5pct"] = rej

# ---- 4) Rank proxies per sensitive attribute ----
# Define thresholds (tune as needed)
thr_cramer = 0.3    # moderate association
thr_pb     = 0.2    # small-to-moderate effect

def flag_proxy(row):
    if pd.isna(row["association"]):
        return False
    if row["metric"] == "cramers_v":
        return (row["association"] >= thr_cramer) and bool(row.get("significant_fdr_5pct", False))
    elif row["metric"].startswith("point_biserial"):
        return (abs(row["association"]) >= thr_pb) and bool(row.get("significant_fdr_5pct", False))
    return False

assoc_df["proxy_flag"] = assoc_df.apply(flag_proxy, axis=1)

# Top candidates to review
proxy_candidates = (assoc_df
                    .sort_values(["sensitive","proxy_flag","association"], ascending=[True, False, False])
                    .groupby("sensitive")
                    .head(25))

print("\n=== Top proxy candidates (by sensitive attribute) ===")
print(proxy_candidates[["sensitive","feature","metric","association","p_adj","proxy_flag"]])

# ---- 5) (Optional) Heatmaps for Cramér’s V within categorical-only space ----
# You can visualize sensitive vs categorical features:
cat_for_heat = [c for c in cat_cols if c != target]
heat = pd.DataFrame(index=sensitive, columns=[c for c in cat_for_heat if c not in sensitive], dtype=float)

for s in sensitive:
    for f in heat.columns: 
        v, _ = cramers_v(df0[s], df0[f])
        heat.loc[s, f] = v

# Example: heat  (plotting left to notebook/figure code as needed)
# import seaborn as sns, matplotlib.pyplot as plt
# plt.figure(figsize=(12, 4))
# sns.heatmap(heat.astype(float), annot=True, vmin=0, vmax=1, cmap="coolwarm")
# plt.title("Cramér’s V: Sensitive vs Categorical (incl. skills)")
# plt.show()

# ---- 6) Save artifacts (tables) ----
assoc_df.to_csv("proxy_bias_associations_all.csv", index=False)
proxy_candidates.to_csv("proxy_bias_candidates_top.csv", index=False)



=== Top proxy candidates (by sensitive attribute) ===
     sensitive             feature                metric  association  \
112  Age_Group              Gender             cramers_v     0.054862   
143  Age_Group        skill__rare_             cramers_v     0.053355   
113  Age_Group                Race             cramers_v     0.040452   
114  Age_Group           Education             cramers_v     0.027505   
165  Age_Group         skill_scrum             cramers_v     0.026578   
..         ...                 ...                   ...          ...   
81        Race            skill__r             cramers_v     0.021922   
62        Race     Screening_Score  point_biserial_o-v-r     0.021868   
85        Race          skill__sql             cramers_v     0.021156   
77        Race  skill__negotiation             cramers_v     0.019992   
97        Race           skill_git             cramers_v     0.019644   

            p_adj  proxy_flag  
112  1.323377e-29       False  
143 